In [43]:
import random as rd

# Matrice de distances entre les differentes villes
distances = [
    [0,1,2,3,4,5,6,1,7],
    [1,0,5,4,3,6,1,9,2],
    [2,5,0,1,6,1,9,3,7],
    [3,4,1,0,5,9,7,2,1],
    [4,3,6,5,0,9,1,2,1],
    [5,6,1,9,9,0,2,1,7],
    [6,1,9,7,1,2,0,4,5],
    [1,9,3,2,2,1,4,0,7],
    [7,2,7,1,1,7,5,7,0]
]

In [ ]:
import numpy as np
import math

filename = r"C:\Users\Sofiane\Desktop\M1I2A\S2\ABI\Projet colonie des abeilles\data\dj38.tsp"

def read_tsp(filename):
    coords = []
    with open(filename, 'r') as file:
        lines = file.readlines()

    node_section = False

    for line in lines:
        line = line.strip()

        # détecter le début des coordonnées
        if line == "NODE_COORD_SECTION":
            node_section = True
            continue

        # détecter la fin
        if line == "EOF":
            break

        # lire les coordonnées
        if node_section:
            parts = line.split()
            if len(parts) >= 3:
                city_id = int(parts[0])
                x = float(parts[1])
                y = float(parts[2])
                coords.append((x, y))

    return coords


def compute_distance_matrix(coords):
    n = len(coords)
    dist_matrix = np.zeros((n, n))

    for i in range(n):
        x1, y1 = coords[i]
        for j in range(n):
            x2, y2 = coords[j]

            dist = math.sqrt((x1 - x2)**2 + (y1 - y2)**2)
            dist_matrix[i][j] = dist

    return dist_matrix

coords = (read_tsp(filename))
distance_matrix = compute_distance_matrix(coords)
nombre_villes = len(coords)

# afficher la matrice
print("Nombre de villes :", nombre_villes)
print("\nMatrice des distances :\n")
print(distance_matrix)

Nombre de villes : 38

Matrice des distances :

[[   0.          290.99301545  794.00183127 ... 1852.35373049
  1624.75194088 1858.09261272]
 [ 290.99301545    0.          512.54097969 ... 1598.94551098
  1412.88752368 1649.18902517]
 [ 794.00183127  512.54097969    0.         ... 1331.29480435
  1288.37008374 1514.19696932]
 ...
 [1852.35373049 1598.94551098 1331.29480435 ...    0.
   440.55907953  444.22745405]
 [1624.75194088 1412.88752368 1288.37008374 ...  440.55907953
     0.          236.48918264]
 [1858.09261272 1649.18902517 1514.19696932 ...  444.22745405
   236.48918264    0.        ]]


In [338]:
def generer_solutions_aleatoire(nbr_solutions, nb_ville):
    solutions = []
    
    for _ in range(0, nbr_solutions):
        solution = rd.sample(range(1,nb_ville),nb_ville-1)
        solution.append(nb_ville)
        solution.insert(0,nb_ville)
        solutions.append(solution)
    
    return solutions

In [348]:
def insert_segment(solution, segment):
    solution1 = solution.copy()
    # print(solution)
    number = solution1[segment[0]]

    for i in range(segment[0], segment[1]):
        solution1[i] = solution1[i+1]

    solution1[segment[1]] = number

    return solution1

In [349]:
def insert(solution):
    solution1 = solution.copy()
    position_ville = rd.randint(1, len(solution)-2)

    ville = solution1.pop(position_ville)
    nouvelle_position = rd.randint(1, len(solution)-2)

    solution1.insert(nouvelle_position, ville)
    return solution1

In [350]:
def calculer_distance(solution, distances):
    d = 0
    for i in range(len(solution) - 1):
        d += distances[solution[i] - 1][solution[i + 1] - 1]
    return d 

def genererSolutionVoisine(solution, flag = 0):
    t = []
    solution1 = solution.copy()

    # Technique swap
    if(flag == 0):
        indexs = rd.sample(range(1, len(solution1)-1), 2)
        s = solution1[indexs[0]]
        solution1[indexs[0]] = solution1[indexs[1]]
        solution1[indexs[1]] = s
        t = solution1.copy()
    # Technique de segment
    if(flag == 1):
        segment = rd.sample(range(1, len(solution1)-1), 2)
        segment.sort()
        t = insert_segment(solution1,segment).copy()
    if(flag == 2):
        t = insert(solution).copy()
    
    return t

In [351]:
def Selection_roulette(fitness):

    total = sum(fitness)
    prob = [f/total for f in fitness] 

    cumulative = []
    s=0
    for p in prob:
        s+=p
        cumulative.append(s)
    
    r = rd.random()

    for i,c in enumerate(cumulative):
        if r <= c :
            return i

In [356]:
def Colonies_abeilles_TSP(max_it, SN, Limit, distance_matrix, nb_ville, flag = 0):
    solutions = generer_solutions_aleatoire(SN, nb_ville)
    trial = []
    fitness = []
    for i in range(SN):
        trial.append(0)
        fitness.append(calculer_distance(solutions[i], distance_matrix))

    Best = min(solutions, key=lambda p: calculer_distance(p, distance_matrix))

    for _ in range(max_it):
        # phase employees
        for i in range(SN):
            solution_v = genererSolutionVoisine(solutions[i], flag)
            if(calculer_distance(solution_v, distance_matrix) < calculer_distance(solutions[i], distance_matrix)):
                solutions[i] = solution_v
                fitness[i] = calculer_distance(solution_v, distance_matrix)
                trial[i] = 0
            else:
                trial[i]+=1
        
        #Phase observatrices
        for i in range(SN):
            index = Selection_roulette(fitness)
            solution_v = genererSolutionVoisine(solutions[index], flag=0)
            if calculer_distance(solution_v, distance_matrix) < calculer_distance(solutions[index], distance_matrix):
                solutions[index] = solution_v
                trial[index] = 0
            else:
                trial[index] += 1

        #Phase Elaireuses
        for i in range(SN):
            if trial[i] > Limit :
                solutions[i] = generer_solutions_aleatoire(1, nombre_villes)[0]
                fitness[i] = calculer_distance(solutions[i], distance_matrix)

        Best = min(solutions, key=lambda p: calculer_distance(p, distance_matrix))
    
    return Best, calculer_distance(Best, distance_matrix)


In [409]:
def GridSearchEval(iterations, SN, Limit, flags, nb_ville):
    results = []

    for t in iterations:
        for s in SN:
            for l in Limit:
                for f in flags:
                    result = {}
                    result["iterations"] = t
                    result["SN"] = s
                    result["Limit"] = l
                    result["flag"] = f
                    sol, fit = Colonies_abeilles_TSP(t, s, l, distance_matrix ,nb_ville ,f)
                    result["best_fitness"] = fit
                    result["best solution"] = sol
                    results.append(result)
    
    best_result = min(results, key=lambda x: x["best_fitness"])
    return best_result

In [ ]:
GridSearchEval([100,200,300,500,1000],[10,20,30,40,50],[5,10,20,30,40], [0,1,2], nombre_villes)